In [ ]:
#Extracting traces from csv, (that comes from calcium imaging data. one cell = one roi = one column)
#Makes a nice report including the image - exports pdf if you want.

#Use: last cell to insert 1.path to csv, 2.avgandroi image and 3.name for pdf report
#start with mode = "show" and then mode = "pdf" if it looks good

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

from scipy.signal import savgol_filter, correlate
from scipy.optimize import curve_fit

from pathlib import Path
import matplotlib.image as mpimg


In [3]:
def load_fiji_csv(path):
    df = pd.read_csv(path)

    # Fiji: manchmal leere Index-Spalte
    if df.columns[0].strip() == "":
        df = df.iloc[:, 1:]

    # ROI-Labels als Index
    if not np.issubdtype(df.iloc[:, 0].dtype, np.number):
        df = df.set_index(df.columns[0])

    df = df.select_dtypes(include=[np.number])

    # Ziel: rows = frames, columns = cells
    if df.shape[0] < df.shape[1]:
        df = df.T

    df = df.reset_index(drop=True)
    return df


In [4]:
def show_traces(
    signals,
    smooth=True,
    height_per_cell=3.0,
    fps=1.0,
    offset_min=0.0,
    bounds_min=(0, 5, 20, 35, 50, 60),
    phase_labels=("Baseline\n(Salt)", "AngII #1", "AngII #2", "AngII #3", "Baseline\n(wash)"),
    show_xlabel_on=(0, -1),
    show_ylabel_on=(0, -1),
    y_label="Fura-2 ratio R340/380",
    avg_img_path=None,
    img_height=3.0,
    meta_height=1.2,
    pdf_out=None, 
    mode="show",
    title_text=None,
    protocol_text=None,
    comment_text=None,
):
    # --- basics / time axis ---
    n_cells = signals.shape[1]
    n_frames = signals.shape[0]
    t_min = np.arange(n_frames) / fps / 60.0

    # --- protocol timing ---
    bounds = [b + offset_min for b in bounds_min]
    mark_lines = bounds[1:-1]

    # --- optional smoothing (display only) ---
    if smooth:
        signals = signals.apply(
            lambda s: savgol_filter(s.to_numpy(), 11, 2),
            axis=0,
            result_type="expand"
        )
        signals.columns = signals.columns

    has_img = avg_img_path is not None

    # --- layout creation (figure is created HERE) ---
    if has_img:
        nrows = n_cells + 2
        fig, axes = plt.subplots(
            nrows, 1,
            figsize=(14, img_height + meta_height + height_per_cell * n_cells),
            constrained_layout=False,
            gridspec_kw={"height_ratios": [img_height, meta_height] + [height_per_cell] * n_cells}
        )
        ax_img, ax_meta = axes[0], axes[1]
        trace_axes = axes[2:]

        img = mpimg.imread(str(avg_img_path))
        ax_img.imshow(img)
        ax_img.axis("off")

        ax_meta.axis("off")
        meta_lines = []
        if protocol_text:
            meta_lines.append(protocol_text)
        if comment_text:
            meta_lines.append(f"Comments: {comment_text}")
        if meta_lines:
            ax_meta.text(
                0.01, 0.98, "\n\n".join(meta_lines),
                transform=ax_meta.transAxes,
                ha="left", va="top", fontsize=10
            )
    else:
        fig, trace_axes = plt.subplots(
            n_cells, 1,
            figsize=(14, height_per_cell * n_cells),
            constrained_layout=False
        )
        if n_cells == 1:
            trace_axes = [trace_axes]

    # --- figure-level title (above image, tight spacing) ---
    if title_text:
        fig.suptitle(title_text, fontsize=14, fontweight="bold", y=0.979)

    if n_cells == 1 and has_img:
        trace_axes = [trace_axes]

    # --- helper for first/last axis addressing ---
    def norm_idx(i):
        return i if i >= 0 else n_cells + i

    xlabel_idx = {norm_idx(i) for i in show_xlabel_on}
    ylabel_idx = {norm_idx(i) for i in show_ylabel_on}

    # --- trace plotting ---
    for i, col in enumerate(signals.columns):
        ax = trace_axes[i]
        y = signals[col].to_numpy()

        ax.plot(t_min, y, lw=1.2)

        y_min, y_max = np.nanmin(y), np.nanmax(y)
        pad = 0.15 * (y_max - y_min) if y_max > y_min else 1
        ax.set_ylim(y_min - pad, y_max + pad)

        for m in mark_lines:
            ax.axvline(m, lw=1, alpha=0.25)

        # ROI label (leftmost, now closer to y-label)
        ax.text(
            -0.10, 0.5, str(col),
            transform=ax.transAxes,
            ha="left", va="center",
            fontsize=10, fontweight="bold",
            clip_on=False
        )

        ax.grid(alpha=0.15)

        # y-label BETWEEN ROI label and plot (tight but readable)
        if i in ylabel_idx:
            ax.set_ylabel(y_label)
            ax.yaxis.set_label_coords(-0.038, 0.5)

        # x-label logic
        if i in xlabel_idx:
            if i == n_cells - 1:
                ax.set_xlabel("time (min)")
            else:
                ax.text(
                    0.5, 0.02, "time (min)",
                    transform=ax.transAxes,
                    ha="center", va="bottom", fontsize=10
                )
        else:
            ax.tick_params(labelbottom=False)

    # --- phase labels on last axis ---
    trace_axes[-1].set_xticks(mark_lines)
    y0, y1 = trace_axes[-1].get_ylim()
    text_y = y0 - 0.20 * (y1 - y0)
    for (a, b), lab in zip(zip(bounds[:-1], bounds[1:]), phase_labels):
        trace_axes[-1].text(
            0.5 * (a + b), text_y, lab,
            ha="center", va="top",
            fontsize=10, clip_on=False
        )

    # --- global spacing ---
    plt.subplots_adjust(
        left=0.20,     # overall left margin
        bottom=0.10,
        hspace=0.25,
        top=0.976      # panels closer to title
    )

    # --- export or show (single-switch) ---
    mode = str(mode).lower()

    if mode in ("pdf", "save"):
        if pdf_out is None:
            raise ValueError("mode='pdf' braucht pdf_out=... (Pfad zur PDF-Datei).")

        pdf_out = Path(pdf_out)
        pdf_out.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(str(pdf_out), format="pdf", dpi=300, bbox_inches="tight")
        plt.close(fig)

        return pdf_out  # praktisch fürs Logging/Batching

    # default: just show (no files written)
    plt.show()
    return None



In [ ]:
from pathlib import Path

csv_path = r"D:\FA_Data_in_processing\csv\12_251110_slice3.csv"   ##ändern
avg_img_path = r"D:\FA_Data_in_processing\cellpose_rois\flat_images\12_flat_slice3_251110.tif"  ##ändern

MODE = "show"  # <--- ÄNDERN: "show" zum anschauen, wenn gut: "pdf"

pdf_out = r"D:\FA_Data_in_processing\trace_reports\12_251110_slice3_traces_plus_image.pdf"

title_text = "12 – Slice3 – 2025/11/10"   
protocol_text = (
    "AngII protocol:\n"
    "5–20 min: 20 pM | 20–35 min: 100 pM | 35–50 min: 500 pM\n"
    "Baseline & wash: 4.5 mM KBBS (no AngII)"
)

comment_text = ""

signals = load_fiji_csv(csv_path)
signals.columns = [f"ROI {i+1}" for i in range(signals.shape[1])]

print("frames x cells:", signals.shape)



out = show_traces(
    signals,
    offset_min=0,
    height_per_cell=3.0,
    show_xlabel_on=(-1, 0),
    show_ylabel_on=(-1, 0),
    avg_img_path=avg_img_path,
    title_text=title_text,
    protocol_text=protocol_text,
    comment_text=comment_text,
    pdf_out=pdf_out,
    mode=MODE,
)
print("Output:", out)


frames x cells: (3864, 30)
Output: D:\FA_Data_in_processing\trace_reports\10_251110_slice4_traces_plus_image.pdf
